In [1]:
import csv
import math
from collections import defaultdict
from pathlib import Path

import numpy as np
import ifcopenshell
import ifcopenshell.geom as geom

# ---------------- Fixed paths ----------------
IFC_PATH = "/home/amja/IFC/IFC_Building/AC20-FZK-Haus_with_SB_IBPSA_P1.ifc"
OUTPUT_DIR = Path("/home/amja/IFC/Output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------- Config ----------------
ORIENT_BINS = ["N", "NE", "E", "SE", "S", "SW", "W", "NW"]
MIN_TRI_AREA = 1e-6  # m²

def clean_text(s):
    if s is None:
        return ""
    return (
        str(s)
        .replace(",", "_")
        .replace("\n", " ")
        .replace("\r", " ")
        .replace("\t", " ")
        .strip()
    )

# ---------------- Geometry helpers ----------------
def unit(v):
    n = np.linalg.norm(v)
    return v / n if n > 0 else v

def tri_area(p0, p1, p2):
    return 0.5 * np.linalg.norm(np.cross(p1 - p0, p2 - p0))

def tri_normal(p0, p1, p2):
    return unit(np.cross(p1 - p0, p2 - p0))

def azimuth_from_normal(n):
    """
    Project normal to XY plane.
    0° at +Y (North), 90° at +X (East).
    """
    horiz = np.array([n[0], n[1], 0.0])
    l = np.linalg.norm(horiz)
    if l < 1e-6:
        return None
    horiz /= l
    ang = math.degrees(math.atan2(horiz[0], horiz[1]))  # atan2(x, y)
    return ang % 360.0

def bin_orientation(angle_deg):
    idx = int(round(angle_deg / 45.0)) % 8
    return ORIENT_BINS[idx]

def make_engine():
    s = geom.settings()
    s.set(geom.settings.USE_WORLD_COORDS, True)
    s.set(geom.settings.SEW_SHELLS, True)
    return s

def mesh_triangles(shape):
    verts = np.array(shape.geometry.verts, dtype=float).reshape(-1, 3)
    faces = np.array(shape.geometry.faces, dtype=int).reshape(-1, 3)
    return verts[faces]

def orientation_bins_and_major_area(shape):
    """
    Return:
      bins_dict: orientation -> sum of triangle areas (vertical faces only)
      major_area: area of the largest bin = 'major surface' area (one side)
    """
    tri = mesh_triangles(shape)
    bins = defaultdict(float)

    for t in tri:
        p0, p1, p2 = t
        A = tri_area(p0, p1, p2)
        if A < MIN_TRI_AREA:
            continue
        n = tri_normal(p0, p1, p2)

        # Ignore mostly horizontal faces (slabs, roofs, ceilings)
        if abs(n[2]) > 0.95:
            continue

        az = azimuth_from_normal(n)
        if az is None:
            continue
        b = bin_orientation(az)
        bins[b] += A

    major_area = max(bins.values()) if bins else 0.0
    return dict(bins), major_area

# ---------------- Storey mapping ----------------
def storey_maps(model):
    """
    Returns:
      product_id_to_storey_id
      storey_id_to_name
      storey_id_to_index (0,1,2… sorted by Elevation)
    """
    p2s = {}
    for rel in model.by_type("IfcRelContainedInSpatialStructure"):
        st = rel.RelatingStructure
        if st and st.is_a("IfcBuildingStorey"):
            sid = st.id()
            for p in rel.RelatedElements or []:
                p2s[p.id()] = sid

    sid_to_name = {}
    sid_to_elev = {}
    for st in model.by_type("IfcBuildingStorey"):
        sid = st.id()
        sid_to_name[sid] = clean_text(getattr(st, "Name", ""))
        elev = getattr(st, "Elevation", None)
        try:
            sid_to_elev[sid] = float(elev) if elev is not None else None
        except Exception:
            sid_to_elev[sid] = None

    known = [(sid, e) for sid, e in sid_to_elev.items() if e is not None]
    unknown = [sid for sid, e in sid_to_elev.items() if e is None]
    known.sort(key=lambda x: x[1])
    order = [sid for sid, _ in known] + unknown
    sid_to_index = {sid: i for i, sid in enumerate(order)}

    return p2s, sid_to_name, sid_to_index

# ---------------- Wall external via boundaries ----------------
def wall_external_flags(model):
    """
    wall_product_id -> bool (True if external)
    """
    flags = defaultdict(lambda: False)
    boundaries = model.by_type("IfcRelSpaceBoundary2ndLevel")
    if not boundaries:
        boundaries = model.by_type("IfcRelSpaceBoundary")

    for rb in boundaries:
        elem = getattr(rb, "RelatedBuildingElement", None)
        if not elem:
            continue
        if not (elem.is_a("IfcWall") or elem.is_a("IfcWallStandardCase")):
            continue

        opp = getattr(rb, "OppositeBoundary", None)
        ext = opp is None or getattr(opp, "RelatingSpace", None) is None
        if ext:
            flags[elem.id()] = True

    return dict(flags)

# ---------------- Window → host wall ----------------
def window_host_wall_gid(model, window):
    """
    Resolve IfcWindow → host IfcWall GlobalId (if possible).
    """
    try:
        invs = model.get_inverse(window)
    except Exception:
        return ""

    openings = []
    for rel in invs:
        if rel.is_a("IfcRelFillsElement") and rel.RelatedBuildingElement == window:
            op = getattr(rel, "RelatingOpeningElement", None)
            if op:
                openings.append(op)

    for op in openings:
        try:
            inv2 = model.get_inverse(op)
        except Exception:
            continue
        for relv in inv2:
            if relv.is_a("IfcRelVoidsElement"):
                wall = getattr(relv, "RelatingBuildingElement", None)
                if wall and (wall.is_a("IfcWall") or wall.is_a("IfcWallStandardCase")):
                    return getattr(wall, "GlobalId", "") or ""
    return ""

# ---------------- Space floor area ----------------
def space_floor_area(model, space):
    """
    Try NetFloorArea / GrossFloorArea / Area from IfcElementQuantity.
    """
    try:
        invs = model.get_inverse(space)
    except Exception:
        invs = []

    preferred = ["NetFloorArea", "GrossFloorArea", "Area"]

    # Pass 1: preferred names
    for rel in invs:
        if not rel.is_a("IfcRelDefinesByProperties"):
            continue
        prop = rel.RelatingPropertyDefinition
        if not prop or not prop.is_a("IfcElementQuantity"):
            continue
        for q in prop.Quantities or []:
            if q.is_a("IfcQuantityArea") and getattr(q, "Name", "") in preferred:
                val = getattr(q, "AreaValue", None)
                if val is not None:
                    try:
                        return float(val)
                    except Exception:
                        pass

    # Pass 2: any IfcQuantityArea
    for rel in invs:
        if not rel.is_a("IfcRelDefinesByProperties"):
            continue
        prop = rel.RelatingPropertyDefinition
        if not prop or not prop.is_a("IfcElementQuantity"):
            continue
        for q in prop.Quantities or []:
            if q.is_a("IfcQuantityArea"):
                val = getattr(q, "AreaValue", None)
                if val is not None:
                    try:
                        return float(val)
                    except Exception:
                        pass
    return None

# ---------------- Wall–space adjacency (no quantities, use major area) ----------------
def wall_space_pairs(model, p2s, sid_to_name, sid_to_index, wall_major_area_by_gid):
    """
    Returns dict[(wall_gid, space_id)] -> (shared_area_m2, storey_id, storey_name, storey_index, space_name)

    We approximate SharedArea_m2 as the wall's "major surface area" (one side of the wall),
    for each adjacent space found via RelSpaceBoundary.
    """
    pairs = {}
    boundaries = model.by_type("IfcRelSpaceBoundary2ndLevel")
    if not boundaries:
        boundaries = model.by_type("IfcRelSpaceBoundary")

    for rb in boundaries:
        space = getattr(rb, "RelatingSpace", None)
        wall  = getattr(rb, "RelatedBuildingElement", None)

        if not space or not wall:
            continue
        if not (wall.is_a("IfcWall") or wall.is_a("IfcWallStandardCase")):
            continue  # only wall–space interfaces

        wall_gid = getattr(wall, "GlobalId", "")
        if not wall_gid:
            continue

        # Get major surface area for this wall (one side)
        major_area = wall_major_area_by_gid.get(wall_gid, 0.0)
        if major_area <= 0:
            continue

        space_id = space.id()
        space_name = clean_text(getattr(space, "Name", f"Space_{space_id}"))

        storey_id = p2s.get(space_id, "")
        storey_name = sid_to_name.get(storey_id, "") if storey_id != "" else ""
        storey_idx = sid_to_index.get(storey_id, "") if storey_id != "" else ""

        key = (wall_gid, space_id)
        # If multiple boundaries exist for the same wall–space pair, keep one (same major area)
        if key not in pairs:
            pairs[key] = (major_area, storey_id, storey_name, storey_idx, space_name)

    return pairs

# ---------------- Main ----------------
def run():
    print("Loading IFC…")
    model = ifcopenshell.open(IFC_PATH)
    settings = make_engine()

    p2s, sid_to_name, sid_to_index = storey_maps(model)
    ext_flags = wall_external_flags(model)

    walls = model.by_type("IfcWall") + model.by_type("IfcWallStandardCase")
    windows = model.by_type("IfcWindow")

    # To later use in wall–space CSV
    wall_major_area_by_gid = {}

    # -------- Elements CSV --------
    elem_csv = OUTPUT_DIR / "elements_geometry.csv"
    with elem_csv.open("w", newline="", encoding="utf-8") as f:
        wcsv = csv.writer(f)
        wcsv.writerow([
            "GlobalId", "Name", "IfcType",
            "StoreyId", "StoreyName", "StoreyIndex",
            "MajorSurfaceArea_m2", "DominantOrientation",
            "N", "NE", "E", "SE", "S", "SW", "W", "NW",
            "IsExternalWall", "HostWallGlobalId",
        ])

        # Walls
        for prod in walls:
            try:
                shape = geom.create_shape(settings, prod)
            except Exception:
                continue

            bins, major_area = orientation_bins_and_major_area(shape)
            dominant = max(bins.items(), key=lambda kv: kv[1])[0] if bins else ""

            sid = p2s.get(prod.id(), "")
            sname = sid_to_name.get(sid, "")
            sidx = sid_to_index.get(sid, "")

            row_bins = [f"{bins.get(b, 0.0):.6f}" for b in ORIENT_BINS]

            gid = getattr(prod, "GlobalId", "")
            wall_major_area_by_gid[gid] = major_area

            wcsv.writerow([
                gid,
                clean_text(getattr(prod, "Name", "")),
                prod.is_a(),
                sid, sname, sidx,
                f"{major_area:.6f}",
                dominant,
                *row_bins,
                "TRUE" if ext_flags.get(prod.id(), False) else "FALSE",
                "",  # HostWallGlobalId not applicable for walls
            ])

        # Windows
        for prod in windows:
            try:
                shape = geom.create_shape(settings, prod)
            except Exception:
                continue

            bins, major_area = orientation_bins_and_major_area(shape)
            dominant = max(bins.items(), key=lambda kv: kv[1])[0] if bins else ""

            sid = p2s.get(prod.id(), "")
            sname = sid_to_name.get(sid, "")
            sidx = sid_to_index.get(sid, "")

            host_gid = window_host_wall_gid(model, prod)
            row_bins = [f"{bins.get(b, 0.0):.6f}" for b in ORIENT_BINS]

            wcsv.writerow([
                getattr(prod, "GlobalId", ""),
                clean_text(getattr(prod, "Name", "")),
                prod.is_a(),
                sid, sname, sidx,
                f"{major_area:.6f}",
                dominant,
                *row_bins,
                "FALSE",
                host_gid,
            ])

    # -------- Spaces CSV --------
    spaces_csv = OUTPUT_DIR / "spaces_floor_areas.csv"
    with spaces_csv.open("w", newline="", encoding="utf-8") as f:
        wcsv = csv.writer(f)
        wcsv.writerow([
            "SpaceId", "SpaceName",
            "StoreyId", "StoreyName", "StoreyIndex",
            "FloorArea_m2",
        ])

        for s in model.by_type("IfcSpace"):
            sid = p2s.get(s.id(), "")
            sname = sid_to_name.get(sid, "")
            sidx = sid_to_index.get(sid, "")

            A = space_floor_area(model, s)
            wcsv.writerow([
                s.id(),
                clean_text(getattr(s, "Name", f"Space_{s.id()}")),
                sid, sname, sidx,
                f"{A:.6f}" if isinstance(A, (int, float)) else "",
            ])

    # -------- Wall–Space shared areas CSV (3rd CSV, geometric approximation) --------
    wall_space_csv = OUTPUT_DIR / "wall_space_shared_areas.csv"
    pairs = wall_space_pairs(model, p2s, sid_to_name, sid_to_index, wall_major_area_by_gid)

    with wall_space_csv.open("w", newline="", encoding="utf-8") as f:
        wcsv = csv.writer(f)
        wcsv.writerow([
            "WallGlobalId",
            "SpaceId", "SpaceName",
            "StoreyId", "StoreyName", "StoreyIndex",
            "SharedArea_m2",
        ])

        for (wall_gid, space_id), (area, storey_id, storey_name, storey_idx, space_name) in pairs.items():
            wcsv.writerow([
                wall_gid,
                space_id,
                space_name,
                storey_id if storey_id is not None else "",
                storey_name,
                storey_idx,
                f"{area:.6f}",
            ])

    print("Saved:")
    print(" -", elem_csv)
    print(" -", spaces_csv)
    print(" -", wall_space_csv)

run()


Loading IFC…
Saved:
 - /home/amja/IFC/Output/elements_geometry.csv
 - /home/amja/IFC/Output/spaces_floor_areas.csv
 - /home/amja/IFC/Output/wall_space_shared_areas.csv


In [3]:
import json
from pathlib import Path

import ifcopenshell

IFC_PATH = "/home/amja/IFC/IFC_Building/AC20-FZK-Haus_with_SB_IBPSA_P1.ifc"
OUT_DIR = Path("/home/amja/IFC/Output")
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_JSON = OUT_DIR / "exterior_walls_selected.json"
OUT_CSV  = OUT_DIR / "exterior_walls_selected.csv"


def safe(v):
    if v is None:
        return ""
    return str(v).replace("\n", " ").replace("\r", " ").strip()


def wall_label_attributes(w):
    """
    'Label attributes' in IFC context typically means the identification-like fields
    (strings / enums) that help a human classify elements.
    """
    # IFC4: IfcWall has PredefinedType (IfcWallTypeEnum) in some exports
    predefined = getattr(w, "PredefinedType", None)
    tag = getattr(w, "Tag", None)
    obj_type = getattr(w, "ObjectType", None)
    name = getattr(w, "Name", None)

    return {
        "id": w.id(),
        "GlobalId": safe(getattr(w, "GlobalId", "")),
        "Name": safe(name),
        "ObjectType": safe(obj_type),
        "Tag": safe(tag),
        "PredefinedType": safe(predefined),
    }


def main():
    model = ifcopenshell.open(IFC_PATH)

    walls = model.by_type("IfcWall") + model.by_type("IfcWallStandardCase")
    rows = [wall_label_attributes(w) for w in walls]

    # Print indexed list for user selection
    print(f"Found {len(rows)} walls.\n")
    print("Index | IFC id | GlobalId | Name | ObjectType | PredefinedType | Tag")
    print("-" * 90)
    for i, r in enumerate(rows):
        print(
            f"{i:5d} | {r['id']:6d} | {r['GlobalId'][:10]:10s} | "
            f"{r['Name'][:25]:25s} | {r['ObjectType'][:18]:18s} | "
            f"{r['PredefinedType'][:12]:12s} | {r['Tag'][:10]:10s}"
        )

    print("\nSelection options:")
    print("  - Enter indices (e.g., 0,1,5-9,22)")
    print("  - Or enter IFC ids prefixed by 'id:' (e.g., id:120,id:121)")
    print("  - Or enter GlobalIds prefixed by 'gid:' (e.g., gid:2bK...,gid:3Xa...)")
    sel = input("\nWhich of these are EXTERIOR walls? > ").strip()

    exterior_ids = set()

    def add_range(a, b):
        for x in range(a, b + 1):
            exterior_ids.add(x)

    # Build quick lookup
    idx_to_ifcid = {i: r["id"] for i, r in enumerate(rows)}
    ifcid_set = {r["id"] for r in rows}
    gid_to_ifcid = {r["GlobalId"]: r["id"] for r in rows if r["GlobalId"]}

    parts = [p.strip() for p in sel.split(",") if p.strip()]
    for p in parts:
        if p.startswith("id:"):
            try:
                wid = int(p.split(":", 1)[1])
                if wid in ifcid_set:
                    exterior_ids.add(wid)
                else:
                    print(f"Warning: IFC id not found among walls: {wid}")
            except Exception:
                print(f"Warning: could not parse {p}")
            continue

        if p.startswith("gid:"):
            gid = p.split(":", 1)[1].strip()
            wid = gid_to_ifcid.get(gid)
            if wid is not None:
                exterior_ids.add(wid)
            else:
                print(f"Warning: GlobalId not found among walls: {gid}")
            continue

        # indices and ranges
        if "-" in p:
            a, b = p.split("-", 1)
            try:
                a = int(a.strip())
                b = int(b.strip())
                for idx in range(min(a, b), max(a, b) + 1):
                    if idx in idx_to_ifcid:
                        exterior_ids.add(idx_to_ifcid[idx])
                    else:
                        print(f"Warning: index out of range: {idx}")
            except Exception:
                print(f"Warning: could not parse range {p}")
        else:
            try:
                idx = int(p)
                if idx in idx_to_ifcid:
                    exterior_ids.add(idx_to_ifcid[idx])
                else:
                    print(f"Warning: index out of range: {idx}")
            except Exception:
                print(f"Warning: could not parse {p}")

    exterior_rows = [r for r in rows if r["id"] in exterior_ids]

    # Save JSON
    payload = {
        "ifc_path": IFC_PATH,
        "ifc_schema": safe(getattr(model, "schema", "")),
        "exterior_wall_count": len(exterior_rows),
        "exterior_walls": exterior_rows,
    }
    OUT_JSON.write_text(json.dumps(payload, indent=2), encoding="utf-8")

    # Save CSV (simple)
    with OUT_CSV.open("w", encoding="utf-8", newline="") as f:
        f.write("IfcId,GlobalId,Name,ObjectType,PredefinedType,Tag\n")
        for r in exterior_rows:
            f.write(
                f"{r['id']},{r['GlobalId']},{r['Name']},{r['ObjectType']},{r['PredefinedType']},{r['Tag']}\n"
            )

    print("\nSaved exterior-wall selection:")
    print(f" - {OUT_JSON}")
    print(f" - {OUT_CSV}")
    print(f"Exterior walls selected: {len(exterior_rows)}")


if __name__ == "__main__":
    main()


Found 26 walls.

Index | IFC id | GlobalId | Name | ObjectType | PredefinedType | Tag
------------------------------------------------------------------------------------------
    0 |  15042 | 2XPyKWY018 | Wand-Int-ERDG-4           |                    |              | BC6F0F70-6
    1 |  17040 | 3PfS__Y_DB | Wand-Int-ERDG-2           |                    |              | 40F78310-9
    2 |  18465 | 2ptk1k7qn8 | Wand-Int-ERDG-1           |                    |              | 8C826359-B
    3 |  18698 | 3jjW3rL656 | Wand-Int-ERDG-3           |                    |              | 2197C9B9-D
    4 |  20598 | 1$wmdwWPjD | Wand-Int-ERDG-5           |                    |              | 623FF5CC-0
    5 |  21966 | 3rPX_Juz59 | Wand-Ext-ERDG-1           |                    |              | BEF1E630-D
    6 |  27421 | 16DNNqzfP2 | Wand-Ext-ERDG-4           |                    |              | A6C3DE63-3
    7 |  31470 | 25fsbPyk15 | Wand-Ext-ERDG-3           |                    |          

In [5]:
import json
from pathlib import Path

import ifcopenshell

IFC_PATH = "/home/amja/IFC/IFC_Building/AC20-FZK-Haus_with_SB_IBPSA_P1.ifc"
OUT_DIR = Path("/home/amja/IFC/Output")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Output from your previous step (the "exterior walls" selection)
EXTERIOR_JSON = OUT_DIR / "exterior_walls_selected.json"

# New output for this step (target walls selection)
TARGET_JSON = OUT_DIR / "target_walls_selected.json"
TARGET_CSV  = OUT_DIR / "target_walls_selected.csv"


def safe(v):
    if v is None:
        return ""
    return str(v).replace("\n", " ").replace("\r", " ").strip()


def load_exterior_wall_ids():
    if not EXTERIOR_JSON.exists():
        raise FileNotFoundError(
            f"Missing {EXTERIOR_JSON}. Run the exterior selection step first."
        )
    data = json.loads(EXTERIOR_JSON.read_text(encoding="utf-8"))
    walls = data.get("exterior_walls", [])
    # stored as 'id' (Ifc entity id)
    return sorted({int(w["id"]) for w in walls if "id" in w})


def get_wall_row(model, wall):
    return {
        "id": wall.id(),
        "GlobalId": safe(getattr(wall, "GlobalId", "")),
        "Name": safe(getattr(wall, "Name", "")),
        "ObjectType": safe(getattr(wall, "ObjectType", "")),
        "Tag": safe(getattr(wall, "Tag", "")),
        "PredefinedType": safe(getattr(wall, "PredefinedType", "")),
        "IfcType": wall.is_a(),
    }


def parse_selection(selection, idx_to_ifcid, ifcid_set, gid_to_ifcid):
    """
    Accept:
      - indices: 0,1,5-9
      - IFC ids: id:120,id:121
      - GlobalIds: gid:2bK...,gid:3Xa...
    Returns set of IfcIds.
    """
    selection = (selection or "").strip()
    parts = [p.strip() for p in selection.split(",") if p.strip()]
    chosen = set()

    for p in parts:
        if p.startswith("id:"):
            try:
                wid = int(p.split(":", 1)[1])
                if wid in ifcid_set:
                    chosen.add(wid)
                else:
                    print(f"Warning: IFC id not found among exterior walls: {wid}")
            except Exception:
                print(f"Warning: could not parse {p}")
            continue

        if p.startswith("gid:"):
            gid = p.split(":", 1)[1].strip()
            wid = gid_to_ifcid.get(gid)
            if wid is not None:
                chosen.add(wid)
            else:
                print(f"Warning: GlobalId not found among exterior walls: {gid}")
            continue

        if "-" in p:
            a, b = p.split("-", 1)
            try:
                a = int(a.strip())
                b = int(b.strip())
                for idx in range(min(a, b), max(a, b) + 1):
                    if idx in idx_to_ifcid:
                        chosen.add(idx_to_ifcid[idx])
                    else:
                        print(f"Warning: index out of range: {idx}")
            except Exception:
                print(f"Warning: could not parse range {p}")
        else:
            try:
                idx = int(p)
                if idx in idx_to_ifcid:
                    chosen.add(idx_to_ifcid[idx])
                else:
                    print(f"Warning: index out of range: {idx}")
            except Exception:
                print(f"Warning: could not parse {p}")

    return chosen


def main():
    model = ifcopenshell.open(IFC_PATH)

    exterior_ifcids = load_exterior_wall_ids()
    if not exterior_ifcids:
        print("No exterior walls found in exterior selection file.")
        return

    # Resolve ids back to entities (filter out anything missing)
    ext_walls = []
    for wid in exterior_ifcids:
        ent = model.by_id(wid)
        if ent and (ent.is_a("IfcWall") or ent.is_a("IfcWallStandardCase")):
            ext_walls.append(ent)

    rows = [get_wall_row(model, w) for w in ext_walls]

    print(f"Exterior walls available: {len(rows)}\n")
    print("Index | IFC id | GlobalId | Name | ObjectType | PredefinedType | Tag")
    print("-" * 90)
    for i, r in enumerate(rows):
        print(
            f"{i:5d} | {r['id']:6d} | {r['GlobalId'][:10]:10s} | "
            f"{r['Name'][:25]:25s} | {r['ObjectType'][:18]:18s} | "
            f"{r['PredefinedType'][:12]:12s} | {r['Tag'][:10]:10s}"
        )

    print("\nSelect TARGET walls from the EXTERIOR list above.")
    print("Selection options:")
    print("  - Indices: 0,1,5-9")
    print("  - IFC ids: id:120,id:121")
    print("  - GlobalIds: gid:2bK...,gid:3Xa...")
    sel = input("\nWhich are TARGET walls? > ").strip()

    idx_to_ifcid = {i: r["id"] for i, r in enumerate(rows)}
    ifcid_set = {r["id"] for r in rows}
    gid_to_ifcid = {r["GlobalId"]: r["id"] for r in rows if r["GlobalId"]}

    target_ids = parse_selection(sel, idx_to_ifcid, ifcid_set, gid_to_ifcid)
    target_rows = [r for r in rows if r["id"] in target_ids]

    payload = {
        "ifc_path": IFC_PATH,
        "ifc_schema": safe(getattr(model, "schema", "")),
        "source_exterior_file": str(EXTERIOR_JSON),
        "target_wall_count": len(target_rows),
        "target_walls": target_rows,
    }
    TARGET_JSON.write_text(json.dumps(payload, indent=2), encoding="utf-8")

    with TARGET_CSV.open("w", encoding="utf-8", newline="") as f:
        f.write("IfcId,GlobalId,Name,ObjectType,PredefinedType,Tag,IfcType\n")
        for r in target_rows:
            f.write(
                f"{r['id']},{r['GlobalId']},{r['Name']},{r['ObjectType']},"
                f"{r['PredefinedType']},{r['Tag']},{r['IfcType']}\n"
            )

    print("\nSaved target-wall selection:")
    print(f" - {TARGET_JSON}")
    print(f" - {TARGET_CSV}")
    print(f"Target walls selected: {len(target_rows)}")


if __name__ == "__main__":
    main()


Exterior walls available: 4

Index | IFC id | GlobalId | Name | ObjectType | PredefinedType | Tag
------------------------------------------------------------------------------------------
    0 |  21966 | 3rPX_Juz59 | Wand-Ext-ERDG-1           |                    |              | BEF1E630-D
    1 |  27421 | 16DNNqzfP2 | Wand-Ext-ERDG-4           |                    |              | A6C3DE63-3
    2 |  31470 | 25fsbPyk15 | Wand-Ext-ERDG-3           |                    |              | D1BD94FD-C
    3 |  32407 | 1bzfVsJqn8 | Wand-Ext-ERDG-2           |                    |              | 74EAE11D-E

Select TARGET walls from the EXTERIOR list above.
Selection options:
  - Indices: 0,1,5-9
  - IFC ids: id:120,id:121
  - GlobalIds: gid:2bK...,gid:3Xa...

Saved target-wall selection:
 - /home/amja/IFC/Output/target_walls_selected.json
 - /home/amja/IFC/Output/target_walls_selected.csv
Target walls selected: 4


In [7]:
# geodata.xlsx generator (IFC4) using geometry-based linking:
# - Wall area: major vertical-face area (m²) from wall geometry (one side)
# - Windows connected to wall: by nearest-wall (geometry) assignment
# - Window area: from IFC quantities/psets if available; auto-convert mm²→m² if needed; else geometry fallback
# - Connected space: by ray/point-in-mesh test using wall outward/indoor normal sampling
# - Zone area: from space geometry (horizontal faces), since Space "Area" attributes are 0
# - Outdoor orientation: single bin (N/NE/...) determined as the wall normal pointing OUTSIDE (away from connected space)
# - Num external walls in space: count of exterior walls (from exterior_walls_selected.json) assigned to that space
# - Total floor area: from IfcSlab quantities if available else slab geometry (upward faces)

import json
import math
from pathlib import Path
from collections import defaultdict

import numpy as np
import ifcopenshell
import ifcopenshell.geom as geom

from openpyxl import Workbook
from openpyxl.styles import Font, Alignment
from openpyxl.utils import get_column_letter

# ---------------- Paths ----------------
IFC_PATH = "/home/amja/IFC/IFC_Building/AC20-FZK-Haus_with_SB_IBPSA_P1.ifc"
OUT_DIR = Path("/home/amja/IFC/Output")
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_JSON = OUT_DIR / "target_walls_selected.json"
EXTERIOR_JSON = OUT_DIR / "exterior_walls_selected.json"

OUT_XLSX = OUT_DIR / "geodata.xlsx"

# ---------------- Config ----------------
ORIENT_BINS = ["N", "NE", "E", "SE", "S", "SW", "W", "NW"]
MIN_TRI_AREA = 1e-6
OFFSET = 0.20  # meters; if classification is noisy, try 0.10–0.35

# If window area properties are in mm², this converts them.
# Heuristic: any "area" value > 200 is assumed mm² (since windows rarely exceed 200 m²)
WINDOW_AREA_MM2_THRESHOLD = 200.0

# ---------------- Small helpers ----------------
def safe(v):
    if v is None:
        return ""
    return str(v).replace("\n", " ").replace("\r", " ").strip()

def to_float(v):
    try:
        if v is None:
            return None
        return float(v)
    except Exception:
        return None

def autosize(ws, min_w=10, max_w=55):
    for col in ws.columns:
        col_letter = get_column_letter(col[0].column)
        max_len = 0
        for cell in col:
            v = cell.value
            if v is None:
                continue
            max_len = max(max_len, len(str(v)))
        ws.column_dimensions[col_letter].width = max(min_w, min(max_w, max_len + 2))

def make_settings():
    s = geom.settings()
    s.set(geom.settings.USE_WORLD_COORDS, True)
    s.set(geom.settings.SEW_SHELLS, True)
    return s

# ---------------- Geometry helpers ----------------
def unit(v):
    n = np.linalg.norm(v)
    return v / n if n > 0 else v

def tri_area(p0, p1, p2):
    return 0.5 * np.linalg.norm(np.cross(p1 - p0, p2 - p0))

def tri_normal(p0, p1, p2):
    return unit(np.cross(p1 - p0, p2 - p0))

def azimuth_from_vec_xy(vxy):
    # 0° at +Y (North), 90° at +X (East)
    x, y = float(vxy[0]), float(vxy[1])
    l = math.hypot(x, y)
    if l < 1e-12:
        return None
    x /= l
    y /= l
    ang = math.degrees(math.atan2(x, y))  # atan2(x,y) => 0=N, 90=E
    return ang % 360.0

def bin_orientation(angle_deg):
    idx = int(round(angle_deg / 45.0)) % 8
    return ORIENT_BINS[idx]

def shape_verts_faces(settings, ent):
    sh = geom.create_shape(settings, ent)
    verts = np.array(sh.geometry.verts, dtype=float).reshape(-1, 3)
    faces = np.array(sh.geometry.faces, dtype=int).reshape(-1, 3)
    return verts, faces

def bbox_of_verts(verts):
    mn = verts.min(axis=0)
    mx = verts.max(axis=0)
    return mn, mx

def point_to_aabb_dist(p, mn, mx):
    # Euclidean distance from point to axis-aligned bbox (0 if inside)
    d = np.maximum(0.0, np.maximum(mn - p, p - mx))
    return float(np.linalg.norm(d))

# ---------------- Robust wall area (one side) ----------------
def wall_major_area_m2(settings, wall):
    verts, faces = shape_verts_faces(settings, wall)
    bins = defaultdict(float)
    for f in faces:
        p0, p1, p2 = verts[f]
        A = tri_area(p0, p1, p2)
        if A < MIN_TRI_AREA:
            continue
        n = tri_normal(p0, p1, p2)
        if abs(n[2]) > 0.95:  # ignore horizontal-ish
            continue
        az = azimuth_from_vec_xy(n[:2])
        if az is None:
            continue
        bins[bin_orientation(az)] += A
    return max(bins.values()) if bins else 0.0

# ---------------- Wall outward orientation (single value) ----------------
def wall_rep_point_and_normals_xy(settings, wall):
    verts, _ = shape_verts_faces(settings, wall)
    p = verts.mean(axis=0)

    xy = verts[:, :2]
    xy = xy - xy.mean(axis=0)

    # PCA on XY to get wall axis, normals = perpendicular to axis
    C = np.cov(xy.T)
    vals, vecs = np.linalg.eigh(C)
    axis = vecs[:, np.argmax(vals)]
    axis = unit(axis)

    n1 = np.array([-axis[1], axis[0]], dtype=float)
    n1 = unit(n1)
    n2 = -n1
    return p, n1, n2

# ---------------- Point-in-mesh test (ray casting) ----------------
def ray_intersect_triangle(orig, direc, v0, v1, v2, eps=1e-9):
    # Möller–Trumbore
    e1 = v1 - v0
    e2 = v2 - v0
    h = np.cross(direc, e2)
    a = float(np.dot(e1, h))
    if -eps < a < eps:
        return None
    f = 1.0 / a
    s = orig - v0
    u = f * float(np.dot(s, h))
    if u < 0.0 or u > 1.0:
        return None
    q = np.cross(s, e1)
    v = f * float(np.dot(direc, q))
    if v < 0.0 or u + v > 1.0:
        return None
    t = f * float(np.dot(e2, q))
    if t > eps:
        return t
    return None

def point_in_mesh(p, verts, faces):
    # Cast a ray along +X. Odd intersections => inside.
    orig = np.array(p, dtype=float)
    direc = np.array([1.0, 0.12345, 0.06789], dtype=float)  # slightly skewed to reduce edge hits
    direc = unit(direc)

    hits = 0
    for f in faces:
        v0, v1, v2 = verts[f]
        t = ray_intersect_triangle(orig, direc, v0, v1, v2)
        if t is not None:
            hits += 1
    return (hits % 2) == 1

# ---------------- Space floor area from space geometry ----------------
def space_floor_area_from_geometry(settings, space):
    verts, faces = shape_verts_faces(settings, space)

    down = 0.0
    up = 0.0
    for f in faces:
        p0, p1, p2 = verts[f]
        A = tri_area(p0, p1, p2)
        if A < MIN_TRI_AREA:
            continue
        n = tri_normal(p0, p1, p2)
        if abs(n[2]) > 0.95:
            if n[2] > 0:
                up += A
            else:
                down += A

    # In a closed volume, up and down should be similar; choose the larger as "floor"
    return float(max(up, down))

# ---------------- Slab total floor area ----------------
def try_get_quantity_area(model, ent):
    # minimal: IfcElementQuantity -> IfcQuantityArea
    try:
        invs = model.get_inverse(ent)
    except Exception:
        invs = []
    for rel in invs:
        if not rel.is_a("IfcRelDefinesByProperties"):
            continue
        qset = rel.RelatingPropertyDefinition
        if not qset or not qset.is_a("IfcElementQuantity"):
            continue
        for q in qset.Quantities or []:
            if q.is_a("IfcQuantityArea"):
                v = to_float(getattr(q, "AreaValue", None))
                if v is not None:
                    return v
    return None

def slab_total_floor_area(model, settings):
    total = 0.0
    slabs = model.by_type("IfcSlab") or []
    any_q = False

    for s in slabs:
        v = try_get_quantity_area(model, s)
        if v is not None:
            total += v
            any_q = True

    if any_q:
        return float(total)

    # geometry fallback: sum upward horizontal-ish faces
    for s in slabs:
        try:
            verts, faces = shape_verts_faces(settings, s)
        except Exception:
            continue
        up = 0.0
        for f in faces:
            p0, p1, p2 = verts[f]
            A = tri_area(p0, p1, p2)
            if A < MIN_TRI_AREA:
                continue
            n = tri_normal(p0, p1, p2)
            if n[2] > 0.95:
                up += A
        total += up
    return float(total)

# ---------------- Window area + conversion ----------------
def window_area_m2(settings, model, win):
    # 1) quantities (if present)
    a = try_get_quantity_area(model, win)
    if a is not None:
        # convert if clearly mm²
        if a > WINDOW_AREA_MM2_THRESHOLD:
            return float(a) / 1e6
        return float(a)

    # 2) OverallWidth * OverallHeight
    w = to_float(getattr(win, "OverallWidth", None))
    h = to_float(getattr(win, "OverallHeight", None))
    if w is not None and h is not None:
        return float(w * h)

    # 3) geometry fallback (major vertical bin area)
    try:
        # For a window, "one side" is fine as its area representation
        return float(wall_major_area_m2(settings, win))
    except Exception:
        return 0.0

# ---------------- Load selected walls ----------------
def load_wall_ids(json_path, key_name):
    data = json.loads(json_path.read_text(encoding="utf-8"))
    arr = data.get(key_name) or []
    return [int(x["id"]) for x in arr if "id" in x]

# ---------------- Main: build meshes, link, write excel ----------------
def main():
    if not TARGET_JSON.exists():
        raise FileNotFoundError(f"Missing {TARGET_JSON}")
    if not EXTERIOR_JSON.exists():
        raise FileNotFoundError(f"Missing {EXTERIOR_JSON}")

    model = ifcopenshell.open(IFC_PATH)
    settings = make_settings()

    target_ids = load_wall_ids(TARGET_JSON, "target_walls")
    exterior_ids = load_wall_ids(EXTERIOR_JSON, "exterior_walls")

    target_walls = []
    for wid in target_ids:
        w = model.by_id(wid)
        if w and (w.is_a("IfcWall") or w.is_a("IfcWallStandardCase")):
            target_walls.append(w)

    exterior_walls = []
    for wid in exterior_ids:
        w = model.by_id(wid)
        if w and (w.is_a("IfcWall") or w.is_a("IfcWallStandardCase")):
            exterior_walls.append(w)

    # -------- Build space meshes + bboxes --------
    spaces = model.by_type("IfcSpace") or []
    space_mesh = {}   # space_id -> (verts, faces, bbox_min, bbox_max)
    space_area = {}   # space_id -> floor area m2 (geometry)
    for sp in spaces:
        try:
            v, f = shape_verts_faces(settings, sp)
        except Exception:
            continue
        mn, mx = bbox_of_verts(v)
        space_mesh[sp.id()] = (v, f, mn, mx)
        space_area[sp.id()] = space_floor_area_from_geometry(settings, sp)

    # -------- Assign a wall to a space using +/- normal point-in-mesh --------
    def wall_connected_space_and_outdoor_bin(wall):
        p, n1_xy, n2_xy = wall_rep_point_and_normals_xy(settings, wall)
        p1 = p + np.array([n1_xy[0], n1_xy[1], 0.0]) * float(OFFSET)
        p2 = p + np.array([n2_xy[0], n2_xy[1], 0.0]) * float(OFFSET)

        # find which space contains p1 or p2 (use bbox prefilter for speed)
        def find_containing_space(pt):
            best = None
            for spid, (v, f, mn, mx) in space_mesh.items():
                # bbox prefilter
                if np.any(pt < (mn - 1e-6)) or np.any(pt > (mx + 1e-6)):
                    continue
                if point_in_mesh(pt, v, f):
                    return spid
            return best

        sp1 = find_containing_space(p1)
        sp2 = find_containing_space(p2)

        # connected space = the "inside" side
        # outdoor normal = opposite of inside
        if sp1 and not sp2:
            connected = sp1
            outdoor_xy = n2_xy
        elif sp2 and not sp1:
            connected = sp2
            outdoor_xy = n1_xy
        elif sp1 and sp2:
            # both inside (thick/overlap); choose the larger area space as connected
            if space_area.get(sp1, 0.0) >= space_area.get(sp2, 0.0):
                connected = sp1
                outdoor_xy = n2_xy
            else:
                connected = sp2
                outdoor_xy = n1_xy
        else:
            # none inside: fallback => choose nearest space bbox
            connected = None
            best_d = 1e18
            for spid, (_, _, mn, mx) in space_mesh.items():
                d = point_to_aabb_dist(p, mn, mx)
                if d < best_d:
                    best_d = d
                    connected = spid
            outdoor_xy = n1_xy

        az = azimuth_from_vec_xy(outdoor_xy)
        outdoor_bin = bin_orientation(az) if az is not None else ""
        return connected, outdoor_bin

    # -------- Precompute: each space -> set of exterior wall labels assigned --------
    # Label to use for linking/output: wall Name like "Surface29"
    ext_walls_by_space = defaultdict(set)
    for w in exterior_walls:
        spid, _ = wall_connected_space_and_outdoor_bin(w)
        if spid:
            ext_walls_by_space[spid].add(safe(getattr(w, "Name", "")))

    # -------- Build wall bboxes for window->wall nearest assignment --------
    wall_bbox = {}  # wall_ifc_id -> (mn, mx)
    for w in target_walls:
        try:
            v, _ = shape_verts_faces(settings, w)
            mn, mx = bbox_of_verts(v)
            wall_bbox[w.id()] = (mn, mx)
        except Exception:
            continue

    # -------- Assign each window to nearest target wall by bbox distance --------
    windows = model.by_type("IfcWindow") or []
    win_to_wall = {}              # window_id -> wall_ifc_id
    wall_to_windows = defaultdict(list)  # wall_ifc_id -> [window]
    for win in windows:
        try:
            v, _ = shape_verts_faces(settings, win)
            p = v.mean(axis=0)
        except Exception:
            continue

        best_wid = None
        best_d = 1e18
        for wid, (mn, mx) in wall_bbox.items():
            d = point_to_aabb_dist(p, mn, mx)
            if d < best_d:
                best_d = d
                best_wid = wid

        # accept only if reasonably close (tune if needed)
        # if your model is meters, 1.0 m is usually safe for window-to-wall proximity
        if best_wid is not None and best_d <= 1.0:
            win_to_wall[win.id()] = best_wid
            wall_to_windows[best_wid].append(win)

    # -------- Total slab floor area --------
    total_floor_A = slab_total_floor_area(model, settings)

    # -------- Write Excel --------
    wb = Workbook()
    ws = wb.active
    ws.title = "geodata"

    headers = [
        "WallLabel",
        "WallGlobalId",
        "WallIfcId",
        "WallArea_m2",
        "ConnectedWindows",
        "WindowsArea_m2",
        "FaceArea_m2",            # FaceArea = WallArea + WindowsArea (per your definition)
        "OutdoorOrientation",
        "ConnectedSpaceId",
        "ConnectedSpaceName",
        "ZoneArea_m2",
        "NumExternalWallsInSpace",
        "TotalFloorArea_m2",
    ]
    ws.append(headers)

    for c in range(1, len(headers) + 1):
        cell = ws.cell(row=1, column=c)
        cell.font = Font(bold=True)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.freeze_panes = "A2"

    for wall in target_walls:
        w_label = safe(getattr(wall, "Name", ""))
        w_gid = safe(getattr(wall, "GlobalId", ""))
        w_ifcid = wall.id()

        # Wall area (m², geometry-based)
        wall_A = float(wall_major_area_m2(settings, wall))

        # Connected space + outward single orientation
        spid, out_orient = wall_connected_space_and_outdoor_bin(wall)
        sp_ent = model.by_id(spid) if spid else None
        sp_name = safe(getattr(sp_ent, "Name", "")) if sp_ent else ""

        # Zone (space) area from geometry
        zone_A = float(space_area.get(spid, 0.0)) if spid else 0.0

        # Windows linked to this wall (nearest bbox assignment)
        wins = wall_to_windows.get(w_ifcid, [])
        win_labels = []
        win_A_sum = 0.0
        for win in wins:
            # label: Name if present else GlobalId
            win_labels.append(safe(getattr(win, "Name", "")) or safe(getattr(win, "GlobalId", "")))
            win_A_sum += float(window_area_m2(settings, model, win))

        # Face area per your definition
        face_A = float(wall_A + win_A_sum)

        # External wall count in this connected space (using same wall->space assignment)
        ext_count = 0
        if spid:
            ext_count = len(ext_walls_by_space.get(spid, set()))

        ws.append([
            w_label,
            w_gid,
            w_ifcid,
            wall_A,
            "; ".join([x for x in win_labels if x]),
            win_A_sum,
            face_A,
            out_orient,
            spid if spid else "",
            sp_name,
            zone_A,
            ext_count,
            total_floor_A,
        ])

    autosize(ws)
    wb.save(OUT_XLSX)
    print(f"Saved: {OUT_XLSX}")

if __name__ == "__main__":
    main()


Saved: /home/amja/IFC/Output/geodata.xlsx
